In [ ]:
import time
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.preprocessing import StandardScaler
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os

import matplotlib.pyplot as plt
import glob
import tensorflow as tf

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import torch
import torch.nn as nn

from torch.autograd import Variable
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import cohen_kappa_score,f1_score
from sklearn.model_selection import KFold, train_test_split
from keras.callbacks import Callback
device = torch.device("cuda")



In [ ]:

# REDUCED_DATA = 250
COMPETITION_DATA_DIR = "/kaggle/input/competitions/rogii-wellbore-geology-prediction"
TRAIN_DIR            = os.path.join(COMPETITION_DATA_DIR, "train")
TEST_DIR             = os.path.join(COMPETITION_DATA_DIR, "test")
SAMPLE_SUBMISSION    = os.path.join(COMPETITION_DATA_DIR, "sample_submission.csv")
SUBMISSION_FILE      = "submission.csv"

WINDOW_SIZE = 64
N_EPOCHS = 20
LEARNING_RATE = 0.001
BATCH_SIZE = 16

wellname = set()

for file in os.listdir(TRAIN_DIR):
    if file[-3:] != "png":
        wellname.add(file.split("__")[0])
wellname = list(wellname)

In [ ]:


class conbr_block(nn.Module):
    def __init__(self, in_layer, out_layer, kernel_size, stride, dilation):
        super(conbr_block, self).__init__()

        self.conv1 = nn.Conv1d(in_layer, out_layer, kernel_size=kernel_size, stride=stride, dilation = dilation, padding = 3, bias=True)
        self.bn = nn.BatchNorm1d(out_layer)
        self.relu = nn.ReLU()
    
    def forward(self,x):
        x = self.conv1(x)
        x = self.bn(x)
        out = self.relu(x)
        
        return out       

class se_block(nn.Module):
    def __init__(self,in_layer, out_layer):
        super(se_block, self).__init__()
        
        self.conv1 = nn.Conv1d(in_layer, out_layer//8, kernel_size=1, padding=0)
        self.conv2 = nn.Conv1d(out_layer//8, in_layer, kernel_size=1, padding=0)
        self.fc = nn.Linear(1,out_layer//8)
        self.fc2 = nn.Linear(out_layer//8,out_layer)
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()
    
    def forward(self,x):

        x_se = nn.functional.adaptive_avg_pool1d(x,1)
        x_se = self.conv1(x_se)
        x_se = self.relu(x_se)
        x_se = self.conv2(x_se)
        x_se = self.sigmoid(x_se)
        
        x_out = torch.add(x, x_se)
        return x_out

class re_block(nn.Module):
    def __init__(self, in_layer, out_layer, kernel_size, dilation):
        super(re_block, self).__init__()
        
        self.cbr1 = conbr_block(in_layer,out_layer, kernel_size, 1, dilation)
        self.cbr2 = conbr_block(out_layer,out_layer, kernel_size, 1, dilation)
        self.seblock = se_block(out_layer, out_layer)
    
    def forward(self,x):

        x_re = self.cbr1(x)
        x_re = self.cbr2(x_re)
        x_re = self.seblock(x_re)
        x_out = torch.add(x, x_re)
        return x_out          




class GeosteeringDualUNet(nn.Module):
    def __init__(self, input_dim, layer_n, kernel_size, typewell_features, num_formations):
        super(GeosteeringDualUNet, self).__init__()
            
        self.input_dim = input_dim
        self.layer_n = layer_n
        self.kernel_size = kernel_size

        # HEAD 1: HORIZONTAL SEQUENCE (4-LAYER DEEP U-NET)
        self.AvgPool1D1 = nn.AvgPool1d(input_dim, stride=5)
        
        # Encoder (Downsampling)
        self.layer1 = self.down_layer(self.input_dim, self.layer_n, self.kernel_size, 1, 2)
        self.layer2 = self.down_layer(self.layer_n, int(self.layer_n*2), self.kernel_size, 5, 2)
        self.layer3 = self.down_layer(int(self.layer_n*2) + int(self.input_dim), int(self.layer_n*3), self.kernel_size, 5, 2)
        self.layer4 = self.down_layer(int(self.layer_n*3), int(self.layer_n*4), self.kernel_size, 5, 2)
        
        # Allows the model to evaluate macro-geology trends across the entire window simultaneously
        self.bottleneck_attn = nn.MultiheadAttention(
            embed_dim=int(self.layer_n*4), 
            num_heads=4, 
            batch_first=True
        )
        
        # Decoder (Upsampling)
        self.upsample = nn.Upsample(scale_factor=5, mode='nearest')
        
        # Input channels: Layer 4 upsampled (layer_n*4) + Layer 3 skip connection (layer_n*3) = layer_n*7
        self.cbr_up1 = conbr_block(int(self.layer_n*7), int(self.layer_n*3), self.kernel_size, 1, 1)
        self.cbr_up2 = conbr_block(int(self.layer_n*5), int(self.layer_n*2), self.kernel_size, 1, 1)
        self.cbr_up3 = conbr_block(int(self.layer_n*3), self.layer_n, self.kernel_size, 1, 1)
        

        # HEAD 2: The Typewell Dictionary (Reference Extractor)
        self.typewell_extractor = nn.Sequential(
            nn.Conv1d(in_channels=typewell_features, out_channels=16, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(in_channels=16, out_channels=32, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1) 
        )

        # OUTPUT HEADS
        merged_channels = self.layer_n + 32

        self.tvt_head = nn.Conv1d(merged_channels, 1, kernel_size=self.kernel_size, stride=1, padding=self.kernel_size//2)
        self.formation_head = nn.Conv1d(merged_channels, num_formations, kernel_size=self.kernel_size, stride=1, padding=self.kernel_size//2)

    def down_layer(self, input_layer, out_layer, kernel, stride, depth):
        block = []
        block.append(conbr_block(input_layer, out_layer, kernel, stride, 1))
        for i in range(depth):
            block.append(re_block(out_layer, out_layer, kernel, 1))
        return nn.Sequential(*block)

    def forward(self, x_horizontal, x_typewell, inference_only=False):

        pool_x1 = self.AvgPool1D1(x_horizontal)
        
        out_0 = self.layer1(x_horizontal)
        out_1 = self.layer2(out_0)

        # Dynamically pad whichever tensor is shorter before concatenating 
        diff = out_1.size(2) - pool_x1.size(2)
        if diff > 0:
            # out_1 is longer, pad pool_x1
            pool_x1 = F.pad(pool_x1, (diff // 2, diff - diff // 2))
        elif diff < 0:
            # pool_x1 is longer, pad out_1
            diff = abs(diff)
            out_1 = F.pad(out_1, (diff // 2, diff - diff // 2))

        
        x_cat = torch.cat([out_1, pool_x1], 1)
        out_2 = self.layer3(x_cat) 
        
        # Pass through the new 4th layer (Deep Bottleneck)
        out_3 = self.layer4(out_2) 
        
        # MultiheadAttention expects (batch, seq_len, channels) when batch_first=True
        out_3_perm = out_3.permute(0, 2, 1)
        attn_out, _ = self.bottleneck_attn(out_3_perm, out_3_perm, out_3_perm)
        out_3 = out_3 + attn_out.permute(0, 2, 1) # Residual connection
        

        # Upsample Layer 4 -> Match and Cat with Layer 3
        up = self.upsample(out_3)
        diff1 = out_2.size(2) - up.size(2)
        up = F.pad(up, (diff1 // 2, diff1 - diff1 // 2))
        up = torch.cat([up, out_2], 1) 
        up = self.cbr_up1(up)
        
        # Upsample -> Match and Cat with Layer 2
        up = self.upsample(up)
        diff2 = out_1.size(2) - up.size(2)
        up = F.pad(up, (diff2 // 2, diff2 - diff2 // 2))
        up = torch.cat([up, out_1], 1) 
        up = self.cbr_up2(up)
        
        # Upsample -> Match and Cat with Layer 1
        up = self.upsample(up)
        diff3 = out_0.size(2) - up.size(2)
        up = F.pad(up, (diff3 // 2, diff3 - diff3 // 2))
        up = torch.cat([up, out_0], 1) 
        unet_features = self.cbr_up3(up) 

        #  PROCESS TYPEWELL
        typewell_features = self.typewell_extractor(x_typewell)
        seq_length = unet_features.shape[2]
        typewell_expanded = typewell_features.expand(-1, -1, seq_length)
        

        # PREDICT
        merged = torch.cat([unet_features, typewell_expanded], dim=1)
        
        tvt_pred = self.tvt_head(merged)             
        
        # If pruning the head during inference to save compute
        if inference_only:
            return tvt_pred
            
        formation_pred = self.formation_head(merged) 
        return tvt_pred, formation_pred

In [ ]:
from torch.utils.data import Dataset

class GeosteeringDataset(Dataset):
    def __init__(self, horizontal_input, well_names, typewell_dict, tvt_target, formation_target):
        self.horizontal_input = horizontal_input
        self.well_names = well_names          # List mapping each window to a well name
        self.typewell_dict = typewell_dict    # The memory-efficient dictionary
        self.tvt_target = tvt_target
        self.formation_target = formation_target
        
    def __len__(self):
        return len(self.horizontal_input)
    
    def __getitem__(self, idx):
        # 1. Get horizontal window
        x_horiz = torch.tensor(np.transpose(self.horizontal_input[idx].copy(), (1,0)), dtype=torch.float)
        
        # 2. Look up the exact Typewell for this specific window!
        well_id = self.well_names[idx]
        type_array = self.typewell_dict[well_id]
        x_type = torch.tensor(np.transpose(type_array.copy(), (1,0)), dtype=torch.float)
        
        # 3. Targets
        y_tvt = torch.tensor(self.tvt_target[idx].copy(), dtype=torch.float)
        y_form = torch.tensor(np.transpose(self.formation_target[idx].copy(), (1,0)), dtype=torch.float)
        
        return x_horiz, x_type, y_tvt, y_form

In [ ]:
def train_model(model, train_loader, valid_loader, epochs=100, lr=0.001,early_stopping_patience=20):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    
    # Load the model weights if a path is provided and the file exists
    if os.path.exists('best_geosteering_model.pt'):
        print(f"Loading saved model...")
        model.load_state_dict(torch.load('best_geosteering_model.pt', map_location=device))

    
    model = model.to(device)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = ReduceLROnPlateau(optimizer, 'min', patience=10, factor=0.8, min_lr=1e-8)
    
    mse_loss = nn.MSELoss()
    
    best_val_loss = float('inf')


    epochs_no_improve = 0
    
    for epoch in range(epochs):
        start_time = time.time()
        model.train()
        train_loss = 0.
        
        for x_horiz, x_type, y_drift, y_form in train_loader:
            x_horiz, x_type = x_horiz.to(device), x_type.to(device)
            y_drift, y_form = y_drift.to(device), y_form.to(device)
            
            # Forward pass
            drift_pred, form_pred = model(x_horiz, x_type)
            drift_pred = drift_pred.squeeze(1)

            # Calculate Losses
            loss_tvt = mse_loss(drift_pred, y_drift)
            loss_form = mse_loss(form_pred, y_form)
            
            # Combined Loss 
            loss = loss_tvt + (0.5 * loss_form)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item() / len(train_loader)
            
        # Validation Loop
        model.eval()
        val_loss = 0.
        val_tvt_rmse = 0.
        
        with torch.no_grad():
            for x_horiz, x_type, y_tvt, y_form in valid_loader:
                x_horiz, x_type = x_horiz.to(device), x_type.to(device)
                y_tvt, y_form = y_tvt.to(device), y_form.to(device)
                
                drift_pred, form_pred = model(x_horiz, x_type)
                drift_pred = drift_pred.squeeze(1)

                loss_tvt = mse_loss(drift_pred, y_tvt)
                loss_form = mse_loss(form_pred, y_form)
                loss = loss_tvt + (0.5 * loss_form)
                val_loss += loss.item() / len(valid_loader)
                
                # FIX: Use y_tvt from the dataloader (which contains your scaled drift target)
                drift_pred_unscaled = drift_pred.cpu().numpy() * scaler_target_drift.scale_[0] + scaler_target_drift.mean_[0]
                y_drift_unscaled = y_tvt.cpu().numpy() * scaler_target_drift.scale_[0] + scaler_target_drift.mean_[0]
                batch_rmse = np.sqrt(np.mean((drift_pred_unscaled - y_drift_unscaled)**2))
                val_tvt_rmse += batch_rmse / len(valid_loader)
                
                        
        scheduler.step(val_loss)
        elapsed = time.time() - start_time
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_no_improve = 0
            torch.save(model.state_dict(), 'best_geosteering_model.pt')

            with open('best_model_stats.txt', 'w') as log_file:
                log_file.write(f"Best Epoch: {epoch + 1}\n")
                log_file.write(f"Train Loss: {train_loss:.4f}\n")
                log_file.write(f"Val Loss: {val_loss:.4f}\n")
                log_file.write(f"Val TVT RMSE: {val_tvt_rmse:.2f} ft\n")
        else:
            epochs_no_improve += 1
        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val TVT RMSE: {val_tvt_rmse:.2f} ft | Time: {elapsed:.1f}s")
        if epochs_no_improve >= early_stopping_patience:
            print(f"\nEarly stopping triggered! No improvement in validation loss for {early_stopping_patience} epochs.")
            break

In [ ]:

import pandas as pd
import numpy as np

import random

def make_sliding_windows(df, feature_cols, target_tvt_col, target_form_col, window_size=64, step=16):
    x_windows, y_tvt_drift_windows, y_form_windows, anchor_vals = [], [], [], []

    for i in range(0, len(df) - window_size + 1, step):
        window = df.iloc[i : i + window_size].copy()

        # Always define a blind_start so we always have an anchor_val
        if random.random() > 0.5:
            blind_start = random.randint(10, window_size - 10)
            anchor_val = window['TVT_input'].iloc[blind_start - 1]
            # Forward-fill the TVT_input to simulate the blind zone
            window.loc[window.index[blind_start:], 'TVT_input'] = anchor_val
        else:
            # If we don't blind it, the anchor is just the last point in the window
            anchor_val = window['TVT_input'].iloc[-1]


        drift_target = window[target_tvt_col].values - anchor_val


        x_windows.append(window[feature_cols].values)
        y_tvt_drift_windows.append(drift_target)
        y_form_windows.append(window[target_form_col].values)
        anchor_vals.append(anchor_val) # Save this for later!

    return np.array(x_windows), np.array(y_tvt_drift_windows), np.array(y_form_windows), np.array(anchor_vals)

features = ['GR', 'Z', 'MD', 'TVT_input'] 
formations = ['ANCC', 'ASTNU', 'ASTNL', 'EGFDU', 'EGFDL', 'BUDA']
    
    new_formation_targets = [f'dist_{form}' for form in formations]
window_size = WINDOW_SIZE
step = 16

max_type_len = 0
for well in wellname:
    t_len = len(pd.read_csv(f"{TRAIN_DIR}/{well}__typewell.csv"))
    if t_len > max_type_len:
        max_type_len = t_len
print(f"Maximum Typewell Length: {max_type_len}")
all_x_horiz = []
all_y_drift = []
all_y_form = []
all_x_anchors = []


features = ['GR', 'Z', 'MD', 'TVT_input']
formations = ['ANCC', 'ASTNU', 'ASTNL', 'EGFDU', 'EGFDL', 'BUDA']

# Keep a single dictionary for typewells, and a list of names for mapping
typewell_dict = {} 
all_well_names = []

for well in wellname:
    horiz_df = pd.read_csv(f"{TRAIN_DIR}/{well}__horizontal_well.csv")
    type_df = pd.read_csv(f"{TRAIN_DIR}/{well}__typewell.csv")

    cols_to_clean = features + formations + ['TVT']
    for col in cols_to_clean:
        horiz_df[col] = horiz_df[col].interpolate().ffill().bfill().fillna(0)
    type_df['GR'] = type_df['GR'].interpolate().ffill().bfill().fillna(0)
    type_df['TVT'] = type_df['TVT'].interpolate().ffill().bfill().fillna(0)
    
    
    x_horiz, y_drift, y_form, anchors = make_sliding_windows(
        horiz_df, feature_cols=features, target_tvt_col='TVT', 
        target_form_col=formations, window_size=window_size, step=16
    )

    # type_df still has its column headers, so this works!
    type_array = type_df[['GR', 'TVT']].values
    
    actual_len = len(type_array)
    if actual_len < max_type_len:
        pad_size = max_type_len - actual_len
        pad_values = np.tile(type_array[-1], (pad_size, 1))
        type_array = np.vstack([type_array, pad_values])

    # STORE ONLY ONE COPY OF THE TYPEWELL in a dictionary
    typewell_dict[well] = type_array
    
    # Store the well name so we know which window belongs to which well
    well_names_for_these_windows = [well for _ in range(len(x_horiz))]

    all_x_horiz.append(x_horiz)
    all_y_drift.append(y_drift)
    all_y_form.append(y_form)
    # all_x_anchors.append(anchors)
    all_well_names.extend([well] * len(x_horiz))


In [ ]:
# # ── How many wells do you have? ──────────────────────────────────────────────
# print(f"Total wells: {len(all_x_horiz)}")
# # each entry is one well's array, so index == well
# current = 0

# for i, (xh, yt, yf) in enumerate(zip(all_x_horiz, all_y_tvt, all_y_form)):
#     print(f"  Well {all_well_names[current+xh.shape[0]-1]}  horiz shape: {xh.shape}")
#     current += xh.shape[0]


In [ ]:
n_wells   = len(wellname)
# n_wells = 10
n_train   = int(n_wells * 0.8)

rng       = np.random.default_rng(seed=42)
well_idx  = rng.permutation(n_wells)
train_idx = well_idx[:n_train]
val_idx   = well_idx[n_train:]

train_input_horiz  = np.concatenate([all_x_horiz[i] for i in train_idx], axis=0)
train_target_drift   = np.concatenate([all_y_drift[i]   for i in train_idx], axis=0)
train_target_form  = np.concatenate([all_y_form[i]  for i in train_idx], axis=0)
train_well_names   = np.concatenate([[wellname[i]] * len(all_x_horiz[i]) for i in train_idx])

val_input_horiz    = np.concatenate([all_x_horiz[i] for i in val_idx],   axis=0)
val_target_drift     = np.concatenate([all_y_drift[i]   for i in val_idx],   axis=0)
val_target_form    = np.concatenate([all_y_form[i]  for i in val_idx],   axis=0)
val_well_names     = np.concatenate([[wellname[i]] * len(all_x_horiz[i]) for i in val_idx])

#  2. Fit scalers on TRAIN only, transform both
scaler_horiz       = StandardScaler()
scaler_target_drift  = StandardScaler()
scaler_target_form = StandardScaler()
scaler_type        = StandardScaler()

shape_h = train_input_horiz.shape
train_input_horiz = scaler_horiz.fit_transform(train_input_horiz.reshape(-1, len(features))).reshape(shape_h)
val_input_horiz   = scaler_horiz.transform(val_input_horiz.reshape(-1, len(features))).reshape(val_input_horiz.shape)

shape_t = train_target_drift.shape
train_target_drift = scaler_target_drift.fit_transform(train_target_drift.reshape(-1, 1)).reshape(shape_t)
val_target_drift   = scaler_target_drift.transform(val_target_drift.reshape(-1, 1)).reshape(val_target_drift.shape)

shape_f = train_target_form.shape
train_target_form = scaler_target_form.fit_transform(train_target_form.reshape(-1, len(formations))).reshape(shape_f)
val_target_form   = scaler_target_form.transform(val_target_form.reshape(-1, len(formations))).reshape(val_target_form.shape)

# Typewell: fit on train wells only
train_well_set = set(wellname[i] for i in train_idx)
all_type_data  = np.vstack([typewell_dict[w] for w in train_well_set])
scaler_type.fit(all_type_data)
for well in typewell_dict:
    typewell_dict[well] = scaler_type.transform(typewell_dict[well])


print(f"Training Windows: {train_input_horiz.shape[0]}")
print(f"Validation Windows: {val_input_horiz.shape[0]}")



In [ ]:
print("NaNs in X:", np.isnan(train_target_form).sum())
print("Infs in X:", np.isinf(train_target_form).sum())

In [ ]:
# Create the Train DataLoader
train_dataset = GeosteeringDataset(
    horizontal_input=train_input_horiz, 
    well_names=train_well_names,           # Your new array
    typewell_dict=typewell_dict,           # Your new dict
    tvt_target=train_target_drift, 
    formation_target=train_target_form
)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

# Create the Validation DataLoader
val_dataset = GeosteeringDataset(
    horizontal_input=val_input_horiz, 
    well_names=val_well_names,             # Your sliced val array
    typewell_dict=typewell_dict,           # Same dict
    tvt_target=val_target_drift, 
    formation_target=val_target_form
)
valid_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
# Initialize the Model 
model = GeosteeringDualUNet(
    input_dim=4,           # Horizontal features: GR, Z, MD, TVT_input
    typewell_features=2,        # Typewell features: GR, TVT
    layer_n=128,           # Base number of filters for the U-Net
    kernel_size=7,         # Size of the 1D convolution window
    # depth=3,               # Number of residual blocks per layer
    num_formations=6       # Number of formation classes to predict (e.g., 6)
)

#  Start Training (removed the extra closing parenthesis)
train_model(model, train_loader, valid_loader, epochs=N_EPOCHS, lr=LEARNING_RATE)


In [ ]:
import pandas as pd
import numpy as np
import torch
import os
import gc
from collections import defaultdict

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
max_type_len = max(len(pd.read_csv(f"{TRAIN_DIR}/{w}__typewell.csv")) for w in wellname)

model = GeosteeringDualUNet(
    input_dim=4,
    typewell_features=2,
    layer_n=128,
    kernel_size=7,
    num_formations=6
)
model = model.to(device)

if os.path.exists('best_geosteering_model.pt'):
    print(f"Loading saved model...")
    model.load_state_dict(torch.load('best_geosteering_model.pt', map_location=device))

model.eval()

#  1. Build the well→row-index map from the sample submission 
sample_sub = pd.read_csv(SAMPLE_SUBMISSION)
well_rows = defaultdict(list)
split = sample_sub['id'].str.rsplit('_', n=1, expand=True)
for well, row_num in zip(split[0], split[1]):
    well_rows[well].append(int(row_num))

pd.DataFrame(columns=['id', 'tvt']).to_csv("submission.csv", index=False)

#  Inference loop 
with torch.no_grad():
    for well in well_rows.keys():
        horiz_df = pd.read_csv(f"{TEST_DIR}/{well}__horizontal_well.csv")
        type_df = pd.read_csv(f"{TEST_DIR}/{well}__typewell.csv")
        
        # Clean NaNs 
        for col in features:
            horiz_df[col] = horiz_df[col].interpolate().ffill().bfill().fillna(0)
        type_df['GR'] = type_df['GR'].interpolate().ffill().bfill().fillna(0)
        type_df['TVT'] = type_df['TVT'].interpolate().ffill().bfill().fillna(0)

        raw_tvt_input = horiz_df['TVT_input'].values.copy()
        
        # USE THE GLOBAL SCALERS (Notice: .transform() ONLY)
        horiz_df[features] = scaler_horiz.transform(horiz_df[features].values)
        type_df[['GR', 'TVT']] = scaler_type.transform(type_df[['GR', 'TVT']].values)
        
                     
        # Pad typewell to max_type_len
        type_array = type_df[['GR', 'TVT']].values
        actual_type_len = len(type_array)
        this_max = max(max_type_len, actual_type_len)   # never shrink below fit baseline
        if actual_type_len < this_max:
            pad = np.tile(type_array[-1], (this_max - actual_type_len, 1))
            type_array = np.vstack([type_array, pad])

        x_type = (torch.tensor(type_array.T, dtype=torch.float)
                       .unsqueeze(0).to(device))

        # Chunk & predict
        num_rows = len(horiz_df)
        predicted_tvt_full = np.zeros(num_rows)
        for i in range(0, num_rows, window_size):
            chunk = horiz_df[features].iloc[i : i + window_size].values
            actual_len = len(chunk)
            anchor_val_for_chunk = raw_tvt_input[i + actual_len - 1]
            if actual_len < window_size:
                chunk = np.vstack([chunk, np.tile(chunk[-1], (window_size - actual_len, 1))])

            x_horiz = (torch.tensor(chunk.T, dtype=torch.float)
                            .unsqueeze(0).to(device))

                        # 1. U-Net predicts the scaled drift
            pred_drift_tensor, _ = model(x_horiz, x_type)
            pred_drift_scaled = pred_drift_tensor.squeeze(1).cpu().numpy()
            pred_drift = scaler_target_drift.inverse_transform(pred_drift_scaled.reshape(-1, 1)).flatten()
            final_tvt_predictions = pred_drift + anchor_val_for_chunk
            predicted_tvt_full[i : i + actual_len] = final_tvt_predictions[:actual_len]
        # Write only the rows Kaggle wants, then free memory immediately
        well_df = pd.DataFrame({
            "id":  [f"{well}_{r}" for r in well_rows[well]],
            "tvt": [predicted_tvt_full[r] for r in well_rows[well]]
        })
        well_df.to_csv("submission.csv", mode='a', header=False, index=False)
        print (well_df)
        del horiz_df, type_df, predicted_tvt_full, well_df, x_type
        gc.collect()

print(f"Done. Total rows: {sum(len(v) for v in well_rows.values())} (expected {len(sample_sub)})")